# CHiRPE Triton ONNX Pipeline Tutorial

This notebook shows how to use the exported ONNX classifier with Triton Inference Server.

You will learn how to:
1. Connect to Triton and inspect model I/O
2. Tokenize text and run classifier inference via Triton
3. Run transcript preprocessing, then score segment summaries through Triton
4. Aggregate segment predictions with voting (majority and average)

## Prerequisites

- Exported model repository exists at `outputs/triton_model_repository/chirpe_classifier/`.
- Triton server is running with that model repository mounted.
- Python packages available: `tritonclient[http]`, `transformers`, `numpy`, `requests`.

Example Triton startup command from repository root:

```bash
docker run --rm --net=host \
  -v ${PWD}/outputs/triton_model_repository:/models \
  nvcr.io/nvidia/tritonserver:24.10-py3 \
  tritonserver --model-repository=/models
```

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import requests
import tritonclient.http as httpclient
from tritonclient.utils import np_to_triton_dtype
from transformers import AutoTokenizer

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "src").exists():
    raise RuntimeError("Run this notebook from repo root or notebooks/.")

sys.path.insert(0, str(REPO_ROOT / "src"))

TRITON_URL = "localhost:8000"
MODEL_NAME = "chirpe_classifier"
MODEL_VERSION = "1"

HF_MODEL_DIR = REPO_ROOT / "outputs" / "test-config-save" / "best_model"
SAMPLE_TRANSCRIPT_PATH = REPO_ROOT / "outputs" / "ultra-quick-test" / "single_transcript.json"


In [ ]:
print(f"HF model dir exists: {HF_MODEL_DIR.exists()} -> {HF_MODEL_DIR}")
print(f"Sample transcript exists: {SAMPLE_TRANSCRIPT_PATH.exists()} -> {SAMPLE_TRANSCRIPT_PATH}")

try:
    ready = requests.get(f"http://{TRITON_URL}/v2/health/ready", timeout=5)
    print(f"Triton ready endpoint: {ready.status_code} ({ready.text.strip()})")
except Exception as exc:
    raise RuntimeError(
        f"Cannot reach Triton at {TRITON_URL}. Start Triton first. Error: {exc}"
    )


In [ ]:
client = httpclient.InferenceServerClient(url=TRITON_URL, verbose=False)

metadata = client.get_model_metadata(
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
)
config = client.get_model_config(
    model_name=MODEL_NAME,
    model_version=MODEL_VERSION,
)

print("Model metadata:")
print(json.dumps(metadata, indent=2))

print("\nModel config:")
print(json.dumps(config, indent=2))


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(HF_MODEL_DIR)

input_names = [item["name"] for item in metadata["inputs"]]
output_name = metadata["outputs"][0]["name"]
label_map = {0: "Healthy", 1: "CHR-P"}

def softmax(logits: np.ndarray) -> np.ndarray:
    shifted = logits - logits.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)

def build_triton_inputs(encoded_batch: dict):
    infer_inputs = []
    input_ids = encoded_batch["input_ids"]

    for name in input_names:
        if name in encoded_batch:
            array = encoded_batch[name]
        elif name == "attention_mask":
            array = np.ones_like(input_ids, dtype=np.int64)
        elif name == "token_type_ids":
            array = np.zeros_like(input_ids, dtype=np.int64)
        else:
            raise ValueError(f"Unsupported Triton input name: {name}")

        array = array.astype(np.int64, copy=False)
        infer_input = httpclient.InferInput(
            name=name,
            shape=array.shape,
            datatype=np_to_triton_dtype(array.dtype),
        )
        infer_input.set_data_from_numpy(array)
        infer_inputs.append(infer_input)

    return infer_inputs

def triton_predict_texts(texts, max_length: int = 128):
    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="np",
    )

    response = client.infer(
        model_name=MODEL_NAME,
        model_version=MODEL_VERSION,
        inputs=build_triton_inputs(encoded),
        outputs=[httpclient.InferRequestedOutput(output_name)],
    )

    logits = response.as_numpy(output_name)
    probabilities = softmax(logits)
    predictions = probabilities.argmax(axis=-1)
    return logits, probabilities, predictions


In [ ]:
sample_texts = [
    "I feel mostly fine and my daily routine is stable.",
    "Sometimes I hear whispers and feel people can read my thoughts.",
    "I am sleeping well and do not notice unusual experiences.",
]

logits, probabilities, predictions = triton_predict_texts(sample_texts)

for idx, text in enumerate(sample_texts):
    pred = int(predictions[idx])
    confidence = float(probabilities[idx, pred])
    print(f"Text #{idx + 1}: {text}")
    print(f"  Predicted label: {label_map[pred]} ({pred})")
    print(f"  Confidence: {confidence:.4f}")
    print(f"  Probabilities [Healthy, CHR-P]: {probabilities[idx].round(4).tolist()}")
    print()


## Run transcript preprocessing + Triton scoring

The Triton model serves classifier inference only.

For an end-to-end CHiRPE flow, do preprocessing locally (segmentation and summarization), then send segment summaries to Triton and aggregate predictions.

In [ ]:
from chirpe.data.preprocessor import TranscriptPreprocessor

with open(SAMPLE_TRANSCRIPT_PATH, "r") as file:
    transcript_item = json.load(file)

preprocessor = TranscriptPreprocessor(
    segmentation_threshold=0.8,
    use_llm_summarizer=False,
)

processed = preprocessor.process_transcript(
    transcript_item["transcript"],
    transcript_item.get("participant_id", "unknown"),
)

segments = [seg for seg in processed.get("segments", []) if seg.get("summary", "").strip()]

print(f"Participant: {processed.get('participant_id')}")
print(f"Raw segments found: {len(processed.get('segments', []))}")
print(f"Non-empty summaries used for Triton: {len(segments)}")

for idx, seg in enumerate(segments[:5], start=1):
    preview = seg['summary'][:140].replace("\n", " ")
    print(f"{idx}. {seg['domain']}: {preview}...")


In [ ]:
segment_summaries = [seg["summary"] for seg in segments]

if not segment_summaries:
    raise RuntimeError("No segment summaries available for inference.")

seg_logits, seg_probabilities, seg_predictions = triton_predict_texts(segment_summaries)

majority_prediction = int(np.bincount(seg_predictions).argmax())
average_prediction = int(np.argmax(seg_probabilities.mean(axis=0)))

result = {
    "participant_id": processed.get("participant_id"),
    "num_segments": len(segment_summaries),
    "majority_voting": {
        "prediction_id": majority_prediction,
        "prediction_label": label_map[majority_prediction],
    },
    "average_voting": {
        "prediction_id": average_prediction,
        "prediction_label": label_map[average_prediction],
        "mean_probabilities": seg_probabilities.mean(axis=0).tolist(),
    },
    "segment_predictions": [
        {
            "domain": seg["domain"],
            "prediction_id": int(seg_predictions[i]),
            "prediction_label": label_map[int(seg_predictions[i])],
            "probabilities": seg_probabilities[i].tolist(),
        }
        for i, seg in enumerate(segments)
    ],
}

print(json.dumps(result, indent=2))


## Next steps

- Wrap `triton_predict_texts` in your service layer.
- Keep segmentation/summarization and voting in application code, while Triton serves only classifier logits.
- For production, use Triton model version pinning and request batching based on your throughput/latency target.